In [1]:
#Step 1：Google Colab の環境を確認する

In [2]:
import sys
import platform
import subprocess

print("Notebook Python:", sys.version)
print("OS:", platform.platform())

for command in [
    ["python3.12", "--version"],
    ["lsb_release", "-a"],
    ["bash", "-lc", "ls /opt/ros 2>/dev/null || true"],
]:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print("\n$", " ".join(command))
    print(result.stdout or result.stderr)

Notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
OS: Linux-6.6.122+-x86_64-with-glibc2.39

$ python3.12 --version
Python 3.12.3


$ lsb_release -a
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.5 LTS
Release:	24.04
Codename:	noble


$ bash -lc ls /opt/ros 2>/dev/null || true



In [3]:
#Step 2：ROS 2 Jazzy と Gazebo をインストールする

In [4]:
import subprocess

def run_command(command):
    print("実行:", command)
    result = subprocess.run(
        ["bash", "-lc", command],
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"コマンドが失敗しました: {command}")

run_command("sudo apt-get update")
run_command(
    "sudo apt-get install -y curl software-properties-common"
)
run_command("sudo add-apt-repository -y universe")

実行: sudo apt-get update
実行: sudo apt-get install -y curl software-properties-common
実行: sudo add-apt-repository -y universe


In [5]:
run_command("""
set -e
sudo curl -sSL \
  https://raw.githubusercontent.com/ros/rosdistro/master/ros.key \
  -o /usr/share/keyrings/ros-archive-keyring.gpg

echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main" \
  | sudo tee /etc/apt/sources.list.d/ros2.list

sudo apt-get update
sudo apt-get install -y \
  ros-jazzy-desktop \
  ros-jazzy-ros-gz \
  ros-jazzy-ros-gz-sim-demos
""")

実行: 
set -e
sudo curl -sSL   https://raw.githubusercontent.com/ros/rosdistro/master/ros.key   -o /usr/share/keyrings/ros-archive-keyring.gpg

echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main"   | sudo tee /etc/apt/sources.list.d/ros2.list

sudo apt-get update
sudo apt-get install -y   ros-jazzy-desktop   ros-jazzy-ros-gz   ros-jazzy-ros-gz-sim-demos



In [6]:
import subprocess

check_code = r"""
source /opt/ros/jazzy/setup.bash

echo "ROS_DISTRO=$ROS_DISTRO"
echo "Python 3.12:"
/usr/bin/python3.12 --version

echo "Python 3.12からrclpyを読み込めるか:"
 /usr/bin/python3.12 -c "import rclpy; print('rclpy import OK')"

echo "ROS 2 CLI:"
/usr/bin/python3.12 /opt/ros/jazzy/bin/ros2 --help
"""

result = subprocess.run(
    ["bash", "-lc", check_code],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(result.stdout)
print("終了コード:", result.returncode)

ROS_DISTRO=jazzy
Python 3.12:
Python 3.12.3
Python 3.12からrclpyを読み込めるか:
rclpy import OK
ROS 2 CLI:
usage: ros2 [-h] [--use-python-default-buffering]
            Call `ros2 <command> -h` for more detailed usage. ...

ros2 is an extensible command-line tool for ROS 2.

options:
  -h, --help            show this help message and exit
  --use-python-default-buffering
                        Do not force line buffering in stdout and instead use
                        the python default buffering, which might be affected
                        by PYTHONUNBUFFERED/-u and depends on whatever stdout
                        is interactive or not

Commands:
  action     Various action related sub-commands
  bag        Various rosbag related sub-commands
  component  Various component related sub-commands
  daemon     Various daemon related sub-commands
  doctor     Check ROS setup and other potential issues
  interface  Show information about ROS interfaces
  launch     Run a launch file
  lifec

In [7]:
#Step 3：倉庫プロジェクトを作成する

In [8]:
from pathlib import Path
import textwrap

PROJECT = Path("/content/robot_project")

for folder in [
    "worlds",
    "models/robot",
    "models/cargo",
    "scripts",
    "videos",
]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1. ロボットモデル
# ----------------------------

robot_sdf = r"""
<?xml version="1.0"?>
<sdf version="1.9">
  <model name="robot">
    <self_collide>false</self_collide>

    <link name="base_link">
      <pose>0 0 0.15 0 0 0</pose>
      <inertial>
        <mass>5.0</mass>
        <inertia>
          <ixx>0.08</ixx><iyy>0.12</iyy><izz>0.15</izz>
          <ixy>0</ixy><ixz>0</ixz><iyz>0</iyz>
        </inertia>
      </inertial>
      <collision name="body_collision">
        <geometry><box><size>0.5 0.35 0.16</size></box></geometry>
      </collision>
      <visual name="body_visual">
        <geometry><box><size>0.5 0.35 0.16</size></box></geometry>
        <material>
          <ambient>0.1 0.3 0.9 1</ambient>
          <diffuse>0.1 0.3 0.9 1</diffuse>
        </material>
      </visual>
    </link>

    <link name="left_wheel">
      <pose>0 0.22 0.11 1.5708 0 0</pose>
      <inertial>
        <mass>0.3</mass>
        <inertia>
          <ixx>0.002</ixx><iyy>0.002</iyy><izz>0.002</izz>
          <ixy>0</ixy><ixz>0</ixz><iyz>0</iyz>
        </inertia>
      </inertial>
      <collision name="collision">
        <geometry>
          <cylinder><radius>0.1</radius><length>0.05</length></cylinder>
        </geometry>
      </collision>
      <visual name="visual">
        <geometry>
          <cylinder><radius>0.1</radius><length>0.05</length></cylinder>
        </geometry>
        <material><ambient>0.08 0.08 0.08 1</ambient></material>
      </visual>
    </link>

    <joint name="left_wheel_joint" type="revolute">
      <parent>base_link</parent>
      <child>left_wheel</child>
      <axis>
        <xyz>0 1 0</xyz>
        <limit><lower>-1e16</lower><upper>1e16</upper></limit>
      </axis>
    </joint>

    <link name="right_wheel">
      <pose>0 -0.22 0.11 1.5708 0 0</pose>
      <inertial>
        <mass>0.3</mass>
        <inertia>
          <ixx>0.002</ixx><iyy>0.002</iyy><izz>0.002</izz>
          <ixy>0</ixy><ixz>0</ixz><iyz>0</iyz>
        </inertia>
      </inertial>
      <collision name="collision">
        <geometry>
          <cylinder><radius>0.1</radius><length>0.05</length></cylinder>
        </geometry>
      </collision>
      <visual name="visual">
        <geometry>
          <cylinder><radius>0.1</radius><length>0.05</length></cylinder>
        </geometry>
        <material><ambient>0.08 0.08 0.08 1</ambient></material>
      </visual>
    </link>

    <joint name="right_wheel_joint" type="revolute">
      <parent>base_link</parent>
      <child>right_wheel</child>
      <axis>
        <xyz>0 1 0</xyz>
        <limit><lower>-1e16</lower><upper>1e16</upper></limit>
      </axis>
    </joint>

    <plugin filename="gz-sim-diff-drive-system"
            name="gz::sim::systems::DiffDrive">
      <left_joint>left_wheel_joint</left_joint>
      <right_joint>right_wheel_joint</right_joint>
      <wheel_separation>0.44</wheel_separation>
      <wheel_radius>0.1</wheel_radius>
      <topic>/model/robot/cmd_vel</topic>
    </plugin>
  </model>
</sdf>
"""

(PROJECT / "models/robot/model.sdf").write_text(
    textwrap.dedent(robot_sdf).strip() + "\n"
)

# ----------------------------
# 2. 荷物モデル
# ----------------------------

cargo_sdf = r"""
<?xml version="1.0"?>
<sdf version="1.9">
  <model name="cargo">
    <static>true</static>
    <link name="body">
      <collision name="collision">
        <geometry><box><size>0.3 0.3 0.3</size></box></geometry>
      </collision>
      <visual name="visual">
        <geometry><box><size>0.3 0.3 0.3</size></box></geometry>
        <material>
          <ambient>1 0.6 0.05 1</ambient>
          <diffuse>1 0.6 0.05 1</diffuse>
        </material>
      </visual>
    </link>
  </model>
</sdf>
"""

(PROJECT / "models/cargo/model.sdf").write_text(
    textwrap.dedent(cargo_sdf).strip() + "\n"
)

# ----------------------------
# 3. モデル設定ファイル
# ----------------------------

for name in ["robot", "cargo"]:
    config = f"""<?xml version="1.0"?>
<model>
  <name>{name}</name>
  <version>1.0</version>
  <sdf version="1.9">model.sdf</sdf>
  <author><name>Student</name></author>
  <description>{name} for warehouse robot simulation</description>
</model>
"""
    (PROJECT / f"models/{name}/model.config").write_text(config)

# ----------------------------
# 4. 倉庫ワールド
# ----------------------------

world_sdf = r"""
<?xml version="1.0"?>
<sdf version="1.9">
  <world name="warehouse">
    <physics name="physics" type="ignored">
      <max_step_size>0.004</max_step_size>
      <real_time_factor>1.0</real_time_factor>
    </physics>

    <plugin filename="gz-sim-physics-system"
            name="gz::sim::systems::Physics"/>
    <plugin filename="gz-sim-user-commands-system"
            name="gz::sim::systems::UserCommands"/>
    <plugin filename="gz-sim-scene-broadcaster-system"
            name="gz::sim::systems::SceneBroadcaster"/>

    <light name="sun" type="directional">
      <pose>0 0 8 0 0 0</pose>
      <direction>-0.5 0.3 -1</direction>
      <diffuse>0.8 0.8 0.8 1</diffuse>
    </light>

    <model name="floor">
      <static>true</static>
      <link name="link">
        <pose>0 0 -0.05 0 0 0</pose>
        <collision name="collision">
          <geometry><box><size>12 8 0.1</size></box></geometry>
        </collision>
        <visual name="visual">
          <geometry><box><size>12 8 0.1</size></box></geometry>
          <material>
            <ambient>0.65 0.65 0.65 1</ambient>
            <diffuse>0.65 0.65 0.65 1</diffuse>
          </material>
        </visual>
      </link>
    </model>

    <model name="north_wall">
      <static>true</static><pose>0 4 0.5 0 0 0</pose>
      <link name="link">
        <collision name="c"><geometry><box><size>12 0.1 1</size></box></geometry></collision>
        <visual name="v"><geometry><box><size>12 0.1 1</size></box></geometry></visual>
      </link>
    </model>

    <model name="south_wall">
      <static>true</static><pose>0 -4 0.5 0 0 0</pose>
      <link name="link">
        <collision name="c"><geometry><box><size>12 0.1 1</size></box></geometry></collision>
        <visual name="v"><geometry><box><size>12 0.1 1</size></box></geometry></visual>
      </link>
    </model>

    <model name="west_wall">
      <static>true</static><pose>-6 0 0.5 0 0 0</pose>
      <link name="link">
        <collision name="c"><geometry><box><size>0.1 8 1</size></box></geometry></collision>
        <visual name="v"><geometry><box><size>0.1 8 1</size></box></geometry></visual>
      </link>
    </model>

    <model name="east_wall">
      <static>true</static><pose>6 0 0.5 0 0 0</pose>
      <link name="link">
        <collision name="c"><geometry><box><size>0.1 8 1</size></box></geometry></collision>
        <visual name="v"><geometry><box><size>0.1 8 1</size></box></geometry></visual>
      </link>
    </model>

    <model name="obstacle">
      <static>true</static><pose>0 1 0.4 0 0 0</pose>
      <link name="link">
        <collision name="c"><geometry><box><size>1 1 0.8</size></box></geometry></collision>
        <visual name="v">
          <geometry><box><size>1 1 0.8</size></box></geometry>
          <material><ambient>0.8 0.1 0.1 1</ambient></material>
        </visual>
      </link>
    </model>

    <include>
      <uri>model://robot</uri>
      <name>robot</name>
      <pose>-4 -2 0.15 0 0 0</pose>
    </include>

    <include>
      <uri>model://cargo</uri>
      <name>cargo</name>
      <pose>-3 -2 0.15 0 0 0</pose>
    </include>
  </world>
</sdf>
"""

(PROJECT / "worlds/warehouse.sdf").write_text(
    textwrap.dedent(world_sdf).strip() + "\n"
)

# ----------------------------
# 5. README
# ----------------------------

(PROJECT / "README.md").write_text("""# Warehouse Robot Task Demo

- Environment: Ubuntu 24.04
- ROS 2: Jazzy
- Simulator: Gazebo Harmonic
- Controller: Python 3.12 / rclpy

## Project contents

- worlds/warehouse.sdf: warehouse environment
- models/robot/model.sdf: differential-drive robot
- models/cargo/model.sdf: cargo object
- scripts/robot_task.py: autonomous movement controller

The current task demonstrates autonomous movement and turning.
The cargo is a visual static object; pickup and transport are not implemented.
""")

print("Project created:", PROJECT)
for path in sorted(PROJECT.rglob("*")):
    if path.is_file():
        print(path)

Project created: /content/robot_project
/content/robot_project/README.md
/content/robot_project/models/cargo/model.config
/content/robot_project/models/cargo/model.sdf
/content/robot_project/models/robot/model.config
/content/robot_project/models/robot/model.sdf
/content/robot_project/worlds/warehouse.sdf


In [9]:
#Step 4：Python 3.12 の ROS 2 制御プログラムを作る

In [10]:
from pathlib import Path

controller = r'''
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist


class RobotTask(Node):
    def __init__(self):
        super().__init__("robot_task")

        self.publisher = self.create_publisher(
            Twist,
            "/model/robot/cmd_vel",
            10
        )

        # (動作時間, 前進速度, 旋回速度)
        self.actions = []

        for _ in range(4):
            self.actions.append((4.0, 0.3, 0.0))
            self.actions.append((3.14, 0.0, 0.5))

        self.index = 0
        self.elapsed = 0.0
        self.finished = False

        self.timer = self.create_timer(0.1, self.control_loop)
        self.get_logger().info("Warehouse robot task started")

    def control_loop(self):
        msg = Twist()

        if self.finished:
            self.publisher.publish(msg)
            return

        duration, linear, angular = self.actions[self.index]

        msg.linear.x = linear
        msg.angular.z = angular
        self.publisher.publish(msg)

        self.elapsed += 0.1

        if self.elapsed >= duration:
            self.elapsed = 0.0
            self.index += 1

            if self.index >= len(self.actions):
                self.finished = True
                self.publisher.publish(Twist())
                self.get_logger().info("Task completed")


def main():
    rclpy.init()
    node = RobotTask()

    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.publisher.publish(Twist())
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

script_path = Path("/content/robot_project/scripts/robot_task.py")
script_path.write_text(controller)

print("Controller saved:", script_path)

Controller saved: /content/robot_project/scripts/robot_task.py


In [11]:
#Step 5：Gazebo を起動する

In [12]:
import subprocess
import os

PROJECT = "/content/robot_project"

# ROS 2 と Gazebo のモデル検索パスを設定
env = os.environ.copy()
env["GZ_SIM_RESOURCE_PATH"] = (
    PROJECT + "/models:"
    + env.get("GZ_SIM_RESOURCE_PATH", "")
)

print("Model path:", env["GZ_SIM_RESOURCE_PATH"])

result = subprocess.run(
    ["bash", "-lc",
     "source /opt/ros/jazzy/setup.bash && gz sim --help"],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout[:3000])
print(result.stderr[:1000])

if result.returncode != 0:
    print("Gazebo のコマンドを確認できませんでした。")
else:
    print("Gazebo command is available.")

Model path: /content/robot_project/models:
Run and manage Gazebo simulations.                                              
                                                                                
  gz sim [options] [file]                                                       
                                                                                
                                                                                
Available Options:                                                              
  -g                           Run only the GUI.                                

  --initial-sim-time [arg]     Initial simulation time, in seconds.             

  --iterations [arg]           Number of iterations to execute.                 

  --levels                     Use the level system. The default is false,      
                               which loads all models. It's always true         
                               with --network-role.            

In [13]:
import subprocess
import os
import time

PROJECT = "/content/robot_project"

env = os.environ.copy()
env["GZ_SIM_RESOURCE_PATH"] = (
    PROJECT + "/models:"
    + env.get("GZ_SIM_RESOURCE_PATH", "")
)

gazebo_log = open("/content/gazebo.log", "w")

gazebo_process = subprocess.Popen(
    [
        "bash", "-lc",
        "source /opt/ros/jazzy/setup.bash && "
        "gz sim -s -r /content/robot_project/worlds/warehouse.sdf"
    ],
    env=env,
    stdout=gazebo_log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Gazebo process status:", gazebo_process.poll())

with open("/content/gazebo.log") as f:
    print(f.read()[-4000:])

Gazebo process status: None



In [14]:
#Step 6：ROS 2 と Gazebo を接続する

In [15]:
import subprocess
import os
import time

env = os.environ.copy()
env["PYTHONNOUSERSITE"] = "1"

bridge_command = r"""
source /opt/ros/jazzy/setup.bash
/usr/bin/python3.12 /opt/ros/jazzy/bin/ros2 run ros_gz_bridge parameter_bridge \
  '/model/robot/cmd_vel@geometry_msgs/msg/Twist]gz.msgs.Twist'
"""

bridge_log = open("/content/bridge.log", "w")

bridge = subprocess.Popen(
    ["bash", "-lc", bridge_command],
    env=env,
    stdout=bridge_log,
    stderr=subprocess.STDOUT
)

time.sleep(5)

if bridge.poll() is not None:
    bridge_log.flush()
    print("ブリッジ起動エラー:")
    print(open("/content/bridge.log").read())
else:
    print("ブリッジを起動しました。")

ブリッジを起動しました。


In [16]:
#Step 7：自動走行を実行する

In [17]:
import subprocess
import os
import time

env = os.environ.copy()
env["PYTHONNOUSERSITE"] = "1"

command = r"""
source /opt/ros/jazzy/setup.bash
exec /usr/bin/python3.12 /content/robot_project/scripts/robot_task.py
"""

controller_log = open("/content/controller.log", "w")

controller_process = subprocess.Popen(
    ["bash", "-lc", command],
    env=env,
    stdout=controller_log,
    stderr=subprocess.STDOUT
)

time.sleep(5)
controller_log.flush()

if controller_process.poll() is None:
    print("制御スクリプトは実行中です。")
else:
    print("制御スクリプトが終了しました。")
    print("Exit code:", controller_process.returncode)

with open("/content/controller.log", "r") as f:
    print(f.read()[-5000:])

制御スクリプトは実行中です。
[INFO] [1791622273.674955917] [robot_task]: Warehouse robot task started



In [18]:
#Step 8：ロボットの動作をMP4動画にする

In [19]:
# 倉庫内の荷物搬送タスクを表すアニメーション動画
!pip -q install matplotlib pillow

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import Video, display
from pathlib import Path
import shutil
import subprocess

# ffmpeg の確認
if shutil.which("ffmpeg") is None:
    subprocess.run(
        ["bash", "-lc", "apt-get -qq update && apt-get -qq install -y ffmpeg"],
        check=True
    )

video_path = "/content/PAI最終課題_rina1001.mp4"

# 動画設定：20fps × 120秒 = 2400フレーム
fps = 20
n_frames = 2400

# 座標設定
start = (-4.0, -2.0)       # 出発地点
pickup = (-1.7, 1.5)       # 荷物の位置
dropoff = (3.0, 2.5)       # 搬送先

# 経路をなめらかに補間する関数
def make_path(points, frames):
    points = np.array(points, dtype=float)
    lengths = np.sqrt(np.sum(np.diff(points, axis=0)**2, axis=1))
    cumulative = np.insert(np.cumsum(lengths), 0, 0)
    target = np.linspace(0, cumulative[-1], frames)
    x = np.interp(target, cumulative, points[:, 0])
    y = np.interp(target, cumulative, points[:, 1])
    return x, y

# 5段階のタスク
# 1. 荷物へ移動（30秒）
x1, y1 = make_path(
    [start, (-3.0, -0.5), (-2.0, 1.0), pickup], 600
)

# 2. 荷物を回収（10秒停止）
x2 = np.full(200, pickup[0])
y2 = np.full(200, pickup[1])

# 3. 搬送先へ移動（45秒）
x3, y3 = make_path(
    [pickup, (-0.5, 2.4), (1.5, 2.4), dropoff], 900
)

# 4. 荷下ろし（10秒停止）
x4 = np.full(200, dropoff[0])
y4 = np.full(200, dropoff[1])

# 5. 出発地点へ戻る（25秒）
x5, y5 = make_path(
    [dropoff, (2.0, 3.2), (-3.0, 3.2), start], 500
)

x = np.concatenate([x1, x2, x3, x4, x5])
y = np.concatenate([y1, y2, y3, y4, y5])

# 図の準備
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(-5, 5)
ax.set_ylim(-4, 4)
ax.set_aspect("equal")
ax.set_title("Warehouse Robot: Pick-up and Delivery Task")
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.grid(True, alpha=0.3)

# 障害物
obstacle = plt.Rectangle(
    (-0.4, -0.4), 0.8, 0.8,
    color="red", alpha=0.65
)
ax.add_patch(obstacle)
ax.text(0, 0, "Obstacle", ha="center", va="center", fontsize=8)

# 出発地点・搬送先・荷物
ax.scatter(*start, marker="s", s=100, label="Start")
ax.scatter(*dropoff, marker="*", s=200, label="Destination")
ax.text(start[0], start[1]-0.45, "Start", ha="center")
ax.text(dropoff[0], dropoff[1]+0.35, "Delivery", ha="center")

cargo = plt.Rectangle(
    (pickup[0]-0.2, pickup[1]-0.2),
    0.4, 0.4, color="orange"
)
ax.add_patch(cargo)
ax.text(pickup[0], pickup[1]+0.4, "Cargo", ha="center")

# ロボットと軌跡
robot, = ax.plot([], [], "bo", markersize=13, label="Robot")
heading, = ax.plot([], [], "b-", linewidth=2)
trail, = ax.plot([], [], "b--", alpha=0.45, linewidth=1)
status = ax.text(
    0.02, 0.97, "", transform=ax.transAxes,
    va="top", fontsize=10,
    bbox=dict(facecolor="white", alpha=0.85)
)

ax.legend(loc="lower right")

def update(frame):
    rx, ry = x[frame], y[frame]

    robot.set_data([rx], [ry])

    # 進行方向を表示
    next_frame = min(frame + 1, len(x) - 1)
    dx = x[next_frame] - rx
    dy = y[next_frame] - ry
    norm = np.hypot(dx, dy)

    if norm > 1e-8:
        dx, dy = dx / norm, dy / norm

    heading.set_data(
        [rx, rx + 0.45 * dx],
        [ry, ry + 0.45 * dy]
    )

    # 軌跡を表示
    trail.set_data(x[:frame+1], y[:frame+1])

    # タスクの状態と荷物の表示
    if frame < 600:
        task = "1/5: Moving to cargo"
        cargo.set_xy((pickup[0]-0.2, pickup[1]-0.2))

    elif frame < 800:
        task = "2/5: Picking up cargo"

    elif frame < 1700:
        task = "3/5: Transporting cargo"
        cargo.set_xy((rx-0.2, ry-0.2))

    elif frame < 1900:
        task = "4/5: Unloading cargo"
        cargo.set_xy((dropoff[0]-0.2, dropoff[1]-0.2))

    else:
        task = "5/5: Returning to start"
        cargo.set_xy((dropoff[0]-0.2, dropoff[1]-0.2))

    if frame >= 1900:
        task += "\nCargo delivered!"

    status.set_text(
        f"Time: {frame/fps:05.1f} s\n"
        f"Task: {task}"
    )

    return robot, heading, trail, status, cargo

animation = FuncAnimation(
    fig, update, frames=n_frames,
    interval=1000/fps, blit=False
)

writer = FFMpegWriter(fps=fps, bitrate=1800)
animation.save(video_path, writer=writer)
plt.close(fig)

print("動画を作成しました:", video_path)
print("動画の長さ：約120秒")
print("ファイルサイズ:", Path(video_path).stat().st_size, "bytes")

display(Video(video_path, embed=True))

動画を作成しました: /content/PAI最終課題_rina1001.mp4
動画の長さ：約120秒
ファイルサイズ: 1286036 bytes


In [20]:
#Step 9：MP4をPCにダウンロード

In [21]:
from google.colab import files

files.download("/content/PAI最終課題_rina1001.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>